# Record an infviz trace on a Colab GPU

This notebook runs the infviz inference engine headless on a Colab GPU. It serves a batch of chat requests through a small Qwen target model with speculative decoding from a smaller draft model, and writes every engine event to a JSONL trace file. The trace is then checked for consistency.

You need a **GPU runtime**: go to **Runtime → Change runtime type** and pick T4, L4, or A100. The output trace can be downloaded and dropped into the infviz web UI to replay the run step by step.


In [ ]:
# Confirm that a GPU is attached
!nvidia-smi


## 1. Upload the engine code

The repo is not on GitHub yet, so upload it as a zip. On your own machine, zip the `engine/` folder (leaving out the virtualenv and cache files) with:

```
cd path/to/infviz && zip -r engine.zip engine -x 'engine/.venv/*' '*/__pycache__/*'
```

Then run the cell below and pick `engine.zip` in the file picker.


In [ ]:
from google.colab import files
files.upload()  # pick engine.zip

# engine.zip contains a top-level engine/ folder, so it unpacks to /content/engine
!unzip -q -o engine.zip -d /content/
%cd /content/engine


## 2. Install infviz

Torch is already installed on Colab, so this only adds the remaining dependencies and the `infviz` command.

In [ ]:
!pip install -q -e .

## 3. Settings

Change these values to adjust the run. The defaults fit a T4 with the 1.5B target.

In [ ]:
# Target model (the model being served). The draft model proposes tokens for it.
TARGET = "Qwen/Qwen2.5-1.5B-Instruct"
# TARGET = "Qwen/Qwen2.5-7B-Instruct"   # needs an L4 or A100; shares the Qwen2.5 tokenizer with the draft

DRAFT = "Qwen/Qwen2.5-0.5B-Instruct"    # use "none" to turn off speculative decoding

N_REQUESTS = 32      # number of chat requests to serve
MEAN_GAP = 2.0       # mean number of engine steps between request arrivals
BLOCKS = 64          # target KV cache size, in blocks
MAX_BATCH = 8        # max requests running at once
SPEC_K = 4           # draft tokens proposed per step
DTYPE = "bf16"       # bf16, fp16, or fp32 (T4 GPUs lack native bf16; try "fp16" there if bf16 is slow or fails)
OUT = "/content/colab_trace.jsonl"


## 4. Record the trace

This runs the workload headless, writes the JSONL trace to `OUT`, and then runs the built-in check on it. The run time depends on the GPU and the settings.

In [ ]:
!infviz record --target {TARGET} --draft {DRAFT} --dtype {DTYPE} --blocks {BLOCKS} --max-batch {MAX_BATCH} --spec-k {SPEC_K} -n {N_REQUESTS} --mean-gap {MEAN_GAP} --seed 0 --out {OUT}


## 5. Download the trace

This saves the `.jsonl` file to your computer. The file is also still available at `/content/colab_trace.jsonl` in this session.

In [ ]:
from google.colab import files
files.download(OUT)


## 6. View the trace in the web UI

1. On your machine, start the web UI: `cd web && npm run dev`, then open the local address Vite prints.
2. Drag the downloaded `.jsonl` file onto the page to replay the run.

Alternatively, copy the file into `web/public/` and open the page with `?trace=/colab_trace.jsonl`. The `?trace=` parameter makes the page fetch the trace from that URL.
